In [11]:
import pandas as pd
import numpy as np
#Load Data
patients_df = pd.read_csv("patient_data.csv")
billing_df = pd.read_csv("billing_data.csv")
patients_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6 entries, 0 to 5
Data columns (total 7 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   PatientID       6 non-null      int64  
 1   Name            6 non-null      object 
 2   Department      6 non-null      object 
 3   Doctor          6 non-null      object 
 4   BillAmount      4 non-null      float64
 5   ReceptionistID  6 non-null      int64  
 6   CheckInTime     6 non-null      object 
dtypes: float64(1), int64(2), object(4)
memory usage: 468.0+ bytes


In [12]:
patients_df = patients_df.drop(columns=['ReceptionistID', 'CheckInTime'], errors='ignore')
patients_df = patients_df[['PatientID', 'Department', 'Doctor', 'BillAmount']]
patients_df = patients_df.drop_duplicates(subset='PatientID')
patients_df['BillAmount'] = patients_df['BillAmount'].fillna(patients_df['BillAmount'].mean())

In [13]:
#Analysis
dept_bill = patients_df.groupby('Department')['BillAmount'].sum()
print(dept_bill)


Department
Cardiology     11200.000000
Dermatology     6233.333333
Neurology       6233.333333
Orthopedics     7500.000000
Name: BillAmount, dtype: float64


In [19]:
#Merge
merged_df = pd.merge(patients_df, billing_df, on='PatientID', how='inner')
print("\n--- Merged Data ---")
print(merged_df.head())


--- Merged Data ---
   PatientID   Department     Doctor   BillAmount  InsuranceCovered  \
0        101   Cardiology  Dr. Smith  5000.000000              2000   
1        102    Neurology   Dr. John  6233.333333              1500   
2        103  Orthopedics    Dr. Lee  7500.000000              2500   
3        104   Cardiology  Dr. Smith  6200.000000              3000   
4        105  Dermatology   Dr. Rose  6233.333333              1000   

   FinalAmount  
0         3000  
1         3500  
2         5000  
3         3200  
4         4000  


In [15]:
#Add new data
new_patients = pd.DataFrame({
    'PatientID': [101, 102],
    'Department': ['Cardiology', 'Neurology'],
    'Doctor': ['Dr.A', 'Dr.B'],
    'BillAmount': [5000, 7000]
})

In [16]:
#Final processing
for col in merged_df.columns:
    if col not in new_patients.columns:
        new_patients[col] = np.nan

new_patients = new_patients[merged_df.columns]
merged_df = pd.concat([merged_df, new_patients], ignore_index=True)

merged_df['InsuranceCovered'] = True
merged_df['FinalAmount'] = merged_df['BillAmount'] * 0.9

In [20]:
#Output
print(merged_df.head())
print("\nDataset Shape:", merged_df.shape)

   PatientID   Department     Doctor   BillAmount  InsuranceCovered  \
0        101   Cardiology  Dr. Smith  5000.000000              2000   
1        102    Neurology   Dr. John  6233.333333              1500   
2        103  Orthopedics    Dr. Lee  7500.000000              2500   
3        104   Cardiology  Dr. Smith  6200.000000              3000   
4        105  Dermatology   Dr. Rose  6233.333333              1000   

   FinalAmount  
0         3000  
1         3500  
2         5000  
3         3200  
4         4000  

Dataset Shape: (5, 6)
